# Run plate_detect tests

Runs the pytest suite under `plate_detection_pipeline/tests`.

- **Fast tests** (default): unit tests, no model download/training.
- **Slow tests** (`slow` marker): full pipeline, downloads `yolov8n.pt` and trains 1 epoch on CPU.

Works locally or on Colab. Run cells top-to-bottom.

## 1. Locate pipeline root + install package

In [1]:
import os
from pathlib import Path

# Notebook lives in <pipeline>/notebooks -> root is parent. Fall back to cwd on Colab.
candidates = [Path.cwd(), Path.cwd().parent]
try:
    candidates.insert(0, Path(__file__).resolve().parent.parent)
except NameError:
    pass

ROOT = next((p for p in candidates if (p / "pyproject.toml").exists() and (p / "tests").is_dir()), None)
assert ROOT is not None, f"plate_detection_pipeline root not found in {candidates}"
os.chdir(ROOT)
print("pipeline root:", ROOT)

pipeline root: /Users/ducqhle/Documents/workspace/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline


In [2]:
# Editable install with dev extras (pytest). Skip if already installed.
import importlib.util, sys
if importlib.util.find_spec("pytest") is None or importlib.util.find_spec("plate_detect") is None:
    !{sys.executable} -m pip install -q -e ".[dev]"

## 2. Fast tests (unit, `not slow`)

In [3]:
!{sys.executable} -m pytest tests -m "not slow" -v

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python3
cachedir: .pytest_cache
rootdir: /Users/ducqhle/Documents/workspace/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 44 items / 1 deselected / 43 selected                                

tests/test_adapters.py::test_reads_polygon_records PASSED                [  2%]
tests/test_bbox.py::test_square_polygon_to_center_wh PASSED              [  4%]
tests/test_bbox.py::test_clamps_out_of_range PASSED                      [  6%]
tests/test_bbox.py::test_degenerate_returns_none PASSED                  [  9%]
tests/test_class_map.py::test_widest_class_is_1row PASSED                [ 11%]
tests/test_class_map.py::test_verify_accepts_matching_bsd_bsv PASSED     [ 13%]
tests/test_class_map.py::test_verify_raises_on_conflict PASSED           [ 16%]
tests/test_cli.py::test

## 3. Slow tests (full pipeline, CPU)

Downloads `yolov8n.pt` on first run and trains 1 epoch. Takes a few minutes. Skip if you only want unit coverage.

In [4]:
!{sys.executable} -m pytest tests -m slow -v

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python3
cachedir: .pytest_cache
rootdir: /Users/ducqhle/Documents/workspace/UIT2026-DoAnCuoiKi/src/ml/plate_detection_pipeline
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 44 items / 43 deselected / 1 selected                                

tests/test_pipeline_smoke.py::test_full_pipeline_cpu PASSED              [100%]

=============================== warnings summary ===============================
tests/test_pipeline_smoke.py::test_full_pipeline_cpu
  /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/ultralytics/utils/export/engine.py:70: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_

## 4. (optional) Whole suite

Run everything in one shot instead of the split cells above.

In [5]:
# !{sys.executable} -m pytest tests -v